In [49]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

In [50]:
cwd = Path.cwd()
project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print('Done!')

In [52]:
from src.get_data_sample import get_sample_data, DATA_ENDPOINT
from src.utils.data_check_functions import api_data_return_check

In [53]:
fixture = pd.read_csv(
    project_root / 'data' / 'england_fixtures_pre_euro_2024.csv',
    parse_dates=['match_date_london']
)

In [54]:
fixture['scheduled_kickoff_utc'] = pd.to_datetime(
    fixture['scheduled_kickoff_utc'], utc=True
)

fixture['scheduled_kickoff_london'] = (
    fixture['scheduled_kickoff_utc'].dt.tz_convert('Europe/London')
)

In [55]:
fixture.head()

,match_id,match_date_london,opponent,england_home_away,competition,scheduled_kickoff_clock,source_timezone,scheduled_kickoff_london,scheduled_kickoff_utc,timing_source_url,inventory_source_url,timing_status,notes
0,ENG_20230323,2023-03-23,Italy,away,UEFA EURO 2024 qualifying,19:45,GMT,2023-03-23 19:45:00+00:00,2023-03-23 19:45:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,NaN
1,ENG_20230326,2023-03-26,Ukraine,home,UEFA EURO 2024 qualifying,17:00,BST,2023-03-26 17:00:00+01:00,2023-03-26 16:00:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,Match-centre preview labels 17:00 GMT; use exp...
2,ENG_20230616,2023-06-16,Malta,away,UEFA EURO 2024 qualifying,19:45,BST,2023-06-16 19:45:00+01:00,2023-06-16 18:45:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,NaN
3,ENG_20230619,2023-06-19,North Macedonia,home,UEFA EURO 2024 qualifying,19:45,BST,2023-06-19 19:45:00+01:00,2023-06-19 18:45:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,Match-centre preview labels 19:45 GMT; advance...
4,ENG_20230909,2023-09-09,Ukraine,away,UEFA EURO 2024 qualifying,17:00,BST,2023-09-09 17:00:00+01:00,2023-09-09 16:00:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,Played in Poland; listed time is British time ...


In [56]:
fixture.shape

(12, 13)

In [57]:
fixture.columns

Index(['match_id', 'match_date_london', 'opponent', 'england_home_away',
       'competition', 'scheduled_kickoff_clock', 'source_timezone',
       'scheduled_kickoff_london', 'scheduled_kickoff_utc',
       'timing_source_url', 'inventory_source_url', 'timing_status', 'notes'],
      dtype='str')

In [58]:
fixture[['match_id', 'match_date_london', 'opponent', 'scheduled_kickoff_london', 'scheduled_kickoff_utc']]

,match_id,match_date_london,opponent,scheduled_kickoff_london,scheduled_kickoff_utc
0,ENG_20230323,2023-03-23,Italy,2023-03-23 19:45:00+00:00,2023-03-23 19:45:00+00:00
1,ENG_20230326,2023-03-26,Ukraine,2023-03-26 17:00:00+01:00,2023-03-26 16:00:00+00:00
2,ENG_20230616,2023-06-16,Malta,2023-06-16 19:45:00+01:00,2023-06-16 18:45:00+00:00
3,ENG_20230619,2023-06-19,North Macedonia,2023-06-19 19:45:00+01:00,2023-06-19 18:45:00+00:00
4,ENG_20230909,2023-09-09,Ukraine,2023-09-09 17:00:00+01:00,2023-09-09 16:00:00+00:00
5,ENG_20230912,2023-09-12,Scotland,2023-09-12 19:45:00+01:00,2023-09-12 18:45:00+00:00
6,ENG_20231013,2023-10-13,Australia,2023-10-13 19:45:00+01:00,2023-10-13 18:45:00+00:00
7,ENG_20231017,2023-10-17,Italy,2023-10-17 19:45:00+01:00,2023-10-17 18:45:00+00:00
8,ENG_20231117,2023-11-17,Malta,2023-11-17 19:45:00+00:00,2023-11-17 19:45:00+00:00
9,ENG_20231120,2023-11-20,North Macedonia,2023-11-20 19:45:00+00:00,2023-11-20 19:45:00+00:00


In [59]:
fixture.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype                        
---  ------                    --------------  -----                        
 0   match_id                  12 non-null     str                          
 1   match_date_london         12 non-null     datetime64[us]               
 2   opponent                  12 non-null     str                          
 3   england_home_away         12 non-null     str                          
 4   competition               12 non-null     str                          
 5   scheduled_kickoff_clock   12 non-null     str                          
 6   source_timezone           12 non-null     str                          
 7   scheduled_kickoff_london  12 non-null     datetime64[us, Europe/London]
 8   scheduled_kickoff_utc     12 non-null     datetime64[us, UTC]          
 9   timing_source_url         12 non-null     str           

In [60]:
len(fixture['match_id'].unique())

12

In [61]:
fixture.iloc[0]['match_date_london']

Timestamp('2023-03-23 00:00:00')

In [62]:
start_time = fixture.iloc[0]['match_date_london']
end_time = start_time + pd.Timedelta(days=1)

In [63]:
start_time, end_time

(Timestamp('2023-03-23 00:00:00'), Timestamp('2023-03-24 00:00:00'))

In [64]:
start_london = start_time.tz_localize('Europe/London')
end_london = end_time.tz_localize('Europe/London')

In [65]:
start_utc = start_london.tz_convert('UTC')
end_utc = end_london.tz_convert('UTC')

In [66]:
start_utc, end_utc

(Timestamp('2023-03-23 00:00:00+0000', tz='UTC'),
 Timestamp('2023-03-24 00:00:00+0000', tz='UTC'))

In [67]:
print(f'Match_id: {fixture.iloc[0]['match_id']}')
print(f'Local Match date start day time: {start_london}')
print(f'Local Match date end day time: {end_london}')
print(f'UTC Match date start day time: {start_utc}')
print(f'UTC Match date end day time: {end_utc}')

Match_id: ENG_20230323
Local Match date start day time: 2023-03-23 00:00:00+00:00
Local Match date end day time: 2023-03-24 00:00:00+00:00
UTC Match date start day time: 2023-03-23 00:00:00+00:00
UTC Match date end day time: 2023-03-24 00:00:00+00:00


In [68]:
params = {
    'from': start_utc.isoformat(),
    'to': end_utc.isoformat(),
    'format': 'json'
}

test_data = get_sample_data(
    endpoint=DATA_ENDPOINT,
    params=params
)

In [71]:
test_data, checks = api_data_return_check(
    data=test_data,
    start_utc=start_utc,
    end_utc=end_utc
)

Converting the data into a dataframe...
Sucessfully converted!
Converting the startTime column in a UTC time stamp...
Successfully converted!
Removing the last index and sorting...
Checking unexpected timestamps...


In [72]:
test_data.head()

,recordType,startTime,demand
0,VD,2023-03-23 00:00:00+00:00,24635
1,VD,2023-03-23 00:05:00+00:00,24323
2,VD,2023-03-23 00:10:00+00:00,24254
3,VD,2023-03-23 00:15:00+00:00,24162
4,VD,2023-03-23 00:20:00+00:00,24091


In [73]:
checks

{'null_timestamp_count_before_filter': np.int64(0),
 'expected_no_of_rows': 288,
 'actual_no_of_rows': 288,
 'missing_timestamp_count': 0,
 'unexpected_timestamp_count': 0,
 'null_demand_count': 0,
 'demand_is_numeric': True,
 'record_type_counts': {'VD': 288},
 'gaps_five_mins': np.True_,
 'duplicate_timestamp_count': np.int64(0)}